# Revised Midterm Project: Campus Facility Reservation & Decision Support System

This revised notebook uses the supplied `facility_reservations (1).xlsx` dataset. The original `midterm.ipynb` is retained separately. The target is **Completed** when `Usage_Status` is `Used`; otherwise it is **Not Completed**.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Prefer the supplied Excel workbook; CSV is the deployment-friendly fallback.
if Path('facility_reservations (1).xlsx').exists():
    df = pd.read_excel('facility_reservations (1).xlsx')
elif Path('facility_reservations.csv').exists():
    df = pd.read_csv('facility_reservations.csv')
else:
    raise FileNotFoundError('Place facility_reservations (1).xlsx or facility_reservations.csv beside this notebook.')
df.columns = df.columns.str.strip()
print('Dataset shape:', df.shape)
display(df.head())

## 1. Data quality and observed overview

In [ ]:
display(df.isna().sum().rename('Missing values').to_frame())
display(df.describe(include='all').T)
print('Usage status counts:')
display(df['Usage_Status'].value_counts(dropna=False).to_frame('Count'))

## 2. Derive target and explore patterns

In [ ]:
df['Reservation_Outcome'] = np.where(
    df['Usage_Status'].astype(str).str.strip().str.lower().eq('used'),
    'Completed', 'Not Completed'
)
print('Outcome distribution:')
display(df['Reservation_Outcome'].value_counts().to_frame('Count'))
print('Outcome by facility type (row percentages):')
display(pd.crosstab(df['Facility_Type'], df['Reservation_Outcome'], normalize='index').mul(100).round(1))
print('Outcome by user category (row percentages):')
display(pd.crosstab(df['User_Category'], df['Reservation_Outcome'], normalize='index').mul(100).round(1))

## 3. Train and evaluate a leakage-aware model

Only fields expected to be known when the reservation is submitted are used as predictors. `Usage_Status`, `Actual_Attendance`, and `Cancellation_Days_Before` are excluded.

In [ ]:
categorical = ['Facility_Type', 'User_Category', 'Reservation_Purpose', 'Reservation_Weekday']
numeric = ['Lead_Time_Days', 'Participants', 'Facility_Capacity', 'Duration_Hours', 'Start_Hour']
features = categorical + numeric
model_df = df[features + ['Reservation_Outcome']].copy()
for col in categorical:
    model_df[col] = model_df[col].astype('string').fillna('Unknown').str.strip()
for col in numeric:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')
X, y = model_df[features], model_df['Reservation_Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = ColumnTransformer([
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric)
])
model = Pipeline([('preprocessor', preprocessor),
                  ('classifier', DecisionTreeClassifier(max_depth=5, min_samples_leaf=8,
                                                        class_weight='balanced', random_state=42))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted precision', 'Weighted recall', 'Weighted F1'],
    'Score': [accuracy_score(y_test, pred),
              precision_score(y_test, pred, average='weighted', zero_division=0),
              recall_score(y_test, pred, average='weighted', zero_division=0),
              f1_score(y_test, pred, average='weighted', zero_division=0)]
})
display(metrics.round(3))
print(classification_report(y_test, pred, zero_division=0))
print('Confusion matrix (label order: Completed, Not Completed):')
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=['Completed', 'Not Completed']),
                     index=['Actual Completed', 'Actual Not Completed'],
                     columns=['Predicted Completed', 'Predicted Not Completed']))

## 4. Example prediction

This is an example only. Adjust the fields to a proposed reservation. The model output is a decision-support estimate, not a guarantee.

In [ ]:
example = pd.DataFrame([{
    'Facility_Type': df['Facility_Type'].mode().iloc[0],
    'User_Category': df['User_Category'].mode().iloc[0],
    'Reservation_Purpose': df['Reservation_Purpose'].mode().iloc[0],
    'Reservation_Weekday': str(df['Reservation_Weekday'].mode().iloc[0]),
    'Lead_Time_Days': float(df['Lead_Time_Days'].median()),
    'Participants': float(df['Participants'].median()),
    'Facility_Capacity': float(df['Facility_Capacity'].median()),
    'Duration_Hours': float(df['Duration_Hours'].median()),
    'Start_Hour': float(df['Start_Hour'].median())
}])
example['Reservation_Weekday'] = example['Reservation_Weekday'].astype('string')
print('Predicted outcome:', model.predict(example[features])[0])
display(pd.DataFrame(model.predict_proba(example[features]), columns=model.named_steps['classifier'].classes_))

## 5. Decision-support notes

- Compare observed rates by facility and user category before proposing interventions.
- Use a predicted high-risk booking to prompt confirmation, not automatic rejection.
- Validate on more recent data and monitor performance over time.
- This prototype has no login, live database integration, or production booking write-back.